# Perbandingan Random Forest dan Weighted FTS

Notebook ini membaca artifact hasil eksperimen Stage 7 dan menampilkan perbandingan Random Forest Direct, Residual FTS-Random Forest, serta Hybrid FTS-Random Forest. Notebook tidak menjalankan training ulang.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results' / 'stage9_arima_sarima'
PREDICTIONS = ROOT / 'results' / 'stage7' / 'stage7_hourly_predictions.csv'
assert RESULTS.exists(), f'Folder hasil tidak ditemukan: {RESULTS}'
assert PREDICTIONS.exists(), f'Prediksi Stage 7 tidak ditemukan: {PREDICTIONS}'
print('Repository:', ROOT)

## 1. Tabel perbandingan Random Forest dan FTS

In [ ]:
models = pd.read_csv(RESULTS / 'ALL_HOURLY_MODELS.csv')
rf_names = [
    'RandomForest Direct',
    'Residual FTS1-RandomForest',
    'Residual FTS2-RandomForest',
    'Residual FTS3-RandomForest',
    'Hybrid FTS1-RandomForest Validation threshold',
    'Hybrid FTS2-RandomForest Validation threshold',
    'Hybrid FTS3-RandomForest Validation threshold',
]
table = models[models['model'].isin(rf_names)][['model', 'validation_daily_mae', 'Daily MAE', 'Hourly MAE']].copy()
table = table.rename(columns={'validation_daily_mae': 'Validation Daily MAE', 'Daily MAE': 'Test Daily MAE', 'Hourly MAE': 'Test Hourly MAE'})
table.sort_values('Test Daily MAE')

## 2. Aktual vs prediksi Residual FTS3-Random Forest

Grafik menggunakan 20 forecast origin test atau 480 prediksi H1-H24.

In [ ]:
pred = pd.read_csv(PREDICTIONS, parse_dates=['origin'])
pred = pred[(pred['model'] == 'Residual FTS3-RandomForest') & (pred['split'] == 'test')].copy()
assert len(pred) == 480, f'Jumlah baris tidak sesuai: {len(pred)}'
pred['target_time'] = pred['origin'] + pd.to_timedelta(pred['horizon'], unit='h')
pred = pred.sort_values('target_time')

plt.figure(figsize=(15, 6))
plt.plot(pred['target_time'], pred['actual'], label='Aktual', color='#244062', linewidth=1.7)
plt.plot(pred['target_time'], pred['predicted'], label='Residual FTS3-RandomForest', color='#d95f02', linewidth=1.5)
plt.title('Aktual vs Prediksi Residual FTS3-RandomForest pada Data Test')
plt.xlabel('Waktu target')
plt.ylabel('Konsumsi energi (kWh)')
plt.grid(axis='y', alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## 3. Interpretasi

Residual FTS3-Random Forest memiliki Test Daily MAE terendah di antara konfigurasi RF yang ditampilkan, tetapi validation-nya buruk. Karena itu hasilnya harus disebut sebagai test-best eksploratif. Random Forest Direct tetap digunakan sebagai baseline RF yang lebih sederhana dan lebih mudah dipertanggungjawabkan.